In [1]:
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(cfg):
    user = cfg["user"]
    pw = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    db = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}?connect_timeout=5",
        pool_pre_ping=True
    )

engine = get_engine(DB_CONFIG)

# 최신 end_day의 upper_outlier 1건 사용 (end_day 미지정이므로 운영상 가장 합리적인 기본값)
SQL_BOUNDARY = text("""
SELECT end_day, upper_outlier
FROM e4_predictive_maintenance.pd_cal_test_ct_summary
WHERE test_contents = :tc
  AND upper_outlier IS NOT NULL
ORDER BY end_day DESC
LIMIT 1
""")

b = pd.read_sql(SQL_BOUNDARY, engine, params={"tc": "1.36_dark_curr_check"})
if b.empty:
    raise RuntimeError("[ERROR] pd_cal_test_ct_summary에서 1.36_dark_curr_check upper_outlier를 찾지 못했습니다.")

boundary_run_time = float(b.loc[0, "upper_outlier"])
boundary_src_day  = str(b.loc[0, "end_day"])

print(f"[OK] boundary_run_time={boundary_run_time} (source end_day={boundary_src_day})")

[OK] boundary_run_time=33.19 (source end_day=2025-11-17)


In [2]:
# 조건
TARGET_END_DAY = "20251219"
TARGET_REMARK  = "PD"

SQL_DATA = text("""
SELECT
    barcode_information,
    remark,
    station,
    end_day,
    end_time,
    run_time
FROM a2_fct_table.fct_table
WHERE end_day = :end_day
  AND remark  = :remark
""")

df = pd.read_sql(
    SQL_DATA,
    engine,
    params={"end_day": TARGET_END_DAY, "remark": TARGET_REMARK}
)

df["run_time"] = pd.to_numeric(df["run_time"], errors="coerce")

# boundary 값 컬럼화
df["boundary_run_time"] = boundary_run_time

# boundary 초과만
df = df[df["run_time"] > df["boundary_run_time"]].copy()

# barcode 1개당 1행
df = (
    df.sort_values(["barcode_information", "end_time"])
      .groupby("barcode_information", as_index=False)
      .agg({
          "remark": "first",
          "station": "first",
          "end_day": "first",
          "end_time": "max",
          "boundary_run_time": "first",
          "run_time": "first"
      })
)

# ============================
# TOP 5% 컷 계산
# ============================
cut = df["run_time"].quantile(0.95)

df_top = df[df["run_time"] >= cut].copy()

df_top = df_top.sort_values("run_time", ascending=False).reset_index(drop=True)

# 출력 컬럼 고정
df_top = df_top[[
    "barcode_information",
    "remark",
    "station",
    "end_day",
    "end_time",
    "boundary_run_time",
    "run_time"
]]

print(f"[OK] boundary={boundary_run_time} / TOP5% cut={cut:.2f} / rows={len(df_top)}")
df_top

[OK] boundary=33.19 / TOP5% cut=39.48 / rows=28


,barcode_information,remark,station,end_day,end_time,boundary_run_time,run_time
0,BA1WJ25350501252USJ8T-14F014-AF,PD,FCT1,20251219,003431,33.19,51.8
1,BA1WJ25352501392USJ8T-14F014-AF,PD,FCT1,20251219,084812,33.19,48.7
2,BA1WJ25352508002USJ8T-14F014-AF,PD,FCT4,20251219,053327,33.19,47.7
3,BA1WJ25351504376USJ8T-14F014-AF,PD,FCT1,20251219,020959,33.19,47.3
4,BA1WJ25351505288USJ8T-14F014-AF,PD,FCT1,20251219,010941,33.19,47.2
5,BA1WJ25352502383USJ8T-14F014-AF,PD,FCT1,20251219,064058,33.19,46.6
6,BA1WJ25352501401USJ8T-14F014-AF,PD,FCT1,20251219,080657,33.19,46.4
7,BA1WJ25352501264USJ8T-14F014-AF,PD,FCT1,20251219,031908,33.19,46.0
8,BA1WJ25352507569USJ8T-14F014-AF,PD,FCT1,20251219,073237,33.19,45.9
9,BA1WJ25350501248USJ8T-14F014-AF,PD,FCT1,20251219,003053,33.19,45.2


In [3]:
# ============================================
# Cell X0) 공통 유틸: run_time 재배치 + 300행 스크롤(상하/좌우) 출력
# ============================================

import pandas as pd
from IPython.display import display, HTML

def _reorder_run_time_after_end_time(df: pd.DataFrame,
                                    run_time_col: str = "run_time",
                                    after_col: str = "end_time") -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df
    cols = list(df.columns)
    if run_time_col not in cols:
        return df
    if after_col not in cols:
        cols.remove(run_time_col)
        cols.insert(0, run_time_col)
        return df[cols]
    cols.remove(run_time_col)
    idx = cols.index(after_col) + 1
    cols.insert(idx, run_time_col)
    return df[cols]

def display_scroll_df(df: pd.DataFrame, n: int = 300, height_px: int = 560,
                      float_format: dict | None = None, title: str | None = None):
    if df is None:
        raise ValueError("[ERROR] df가 None 입니다.")
    view = df.head(n).copy()

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)

    sty = view.style
    if float_format:
        sty = sty.format(float_format, na_rep="")

    html = sty.to_html()
    wrapper = f"""
    <div style="border:1px solid #ddd; padding:8px;">
      {f"<div style='font-weight:600; margin-bottom:6px;'>{title}</div>" if title else ""}
      <div style="overflow:auto; width:100%; max-height:{int(height_px)}px;">
        {html}
      </div>
    </div>
    """
    display(HTML(wrapper))

print("[OK] Cell X0 loaded")

[OK] Cell X0 loaded


In [4]:
# ============================================
# Cell A) boundary_run_time 로드 (당신이 올린 1st cell 유지)
# ============================================

import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(cfg):
    user = cfg["user"]
    pw = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    db = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}?connect_timeout=5",
        pool_pre_ping=True
    )

engine = get_engine(DB_CONFIG)

SQL_BOUNDARY = text("""
SELECT end_day, upper_outlier
FROM e4_predictive_maintenance.pd_cal_test_ct_summary
WHERE test_contents = :tc
  AND upper_outlier IS NOT NULL
ORDER BY end_day DESC
LIMIT 1
""")

b = pd.read_sql(SQL_BOUNDARY, engine, params={"tc": "1.36_dark_curr_check"})
if b.empty:
    raise RuntimeError("[ERROR] pd_cal_test_ct_summary에서 1.36_dark_curr_check upper_outlier를 찾지 못했습니다.")

boundary_run_time = float(b.loc[0, "upper_outlier"])
boundary_src_day  = str(b.loc[0, "end_day"])

print(f"[OK] boundary_run_time={boundary_run_time} (source end_day={boundary_src_day})")

[OK] boundary_run_time=33.19 (source end_day=2025-11-17)


In [5]:
# ============================================
# Cell B) TOP 5% 산출 (당신이 올린 2nd cell 유지 + 출력 스크롤만 적용)
#  - 결과: df_top (run_time 포함)
# ============================================

from sqlalchemy import text
import pandas as pd

TARGET_END_DAY = "20251219"
TARGET_REMARK  = "PD"

SQL_DATA = text("""
SELECT
    barcode_information,
    remark,
    station,
    end_day,
    end_time,
    run_time
FROM a2_fct_table.fct_table
WHERE end_day = :end_day
  AND remark  = :remark
""")

df = pd.read_sql(SQL_DATA, engine, params={"end_day": TARGET_END_DAY, "remark": TARGET_REMARK})
df["run_time"] = pd.to_numeric(df["run_time"], errors="coerce")

df["boundary_run_time"] = boundary_run_time
df = df[df["run_time"] > df["boundary_run_time"]].copy()

df = (
    df.sort_values(["barcode_information", "end_time"])
      .groupby("barcode_information", as_index=False)
      .agg({
          "remark": "first",
          "station": "first",
          "end_day": "first",
          "end_time": "max",
          "boundary_run_time": "first",
          "run_time": "first"
      })
)

cut = df["run_time"].quantile(0.95)

df_top = df[df["run_time"] >= cut].copy()
df_top = df_top.sort_values("run_time", ascending=False).reset_index(drop=True)

df_top = df_top[[
    "barcode_information",
    "remark",
    "station",
    "end_day",
    "end_time",
    "boundary_run_time",
    "run_time"
]]

print(f"[OK] boundary={boundary_run_time} / TOP5% cut={cut:.2f} / rows={len(df_top)}")

df_top = _reorder_run_time_after_end_time(df_top, "run_time", "end_time")
display_scroll_df(
    df_top, n=300, height_px=520,
    float_format={"boundary_run_time":"{:.2f}", "run_time":"{:.2f}"},
    title="df_top (TOP 5%, top 300, scroll)"
)

[OK] boundary=33.19 / TOP5% cut=39.48 / rows=28


,barcode_information,remark,station,end_day,end_time,run_time,boundary_run_time
0,BA1WJ25350501252USJ8T-14F014-AF,PD,FCT1,20251219,003431,51.80,33.19
1,BA1WJ25352501392USJ8T-14F014-AF,PD,FCT1,20251219,084812,48.70,33.19
2,BA1WJ25352508002USJ8T-14F014-AF,PD,FCT4,20251219,053327,47.70,33.19
3,BA1WJ25351504376USJ8T-14F014-AF,PD,FCT1,20251219,020959,47.30,33.19
4,BA1WJ25351505288USJ8T-14F014-AF,PD,FCT1,20251219,010941,47.20,33.19
5,BA1WJ25352502383USJ8T-14F014-AF,PD,FCT1,20251219,064058,46.60,33.19
6,BA1WJ25352501401USJ8T-14F014-AF,PD,FCT1,20251219,080657,46.40,33.19
7,BA1WJ25352501264USJ8T-14F014-AF,PD,FCT1,20251219,031908,46.00,33.19
8,BA1WJ25352507569USJ8T-14F014-AF,PD,FCT1,20251219,073237,45.90,33.19
9,BA1WJ25350501248USJ8T-14F014-AF,PD,FCT1,20251219,003053,45.20,33.19


In [6]:
# ============================================
# Cell X1) df_top 바코드 → fct_detail 조회 + station/run_time/boundary_run_time merge
#  - 핵심: df_top의 run_time을 그대로 가져와서 df_detail에 붙임
# ============================================

import pandas as pd
from sqlalchemy import text

if "df_top" not in globals() or len(df_top) == 0:
    raise ValueError("[ERROR] df_top이 없거나 비어있습니다. Cell B를 먼저 실행하세요.")

# barcode list
barcodes = (
    df_top["barcode_information"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)
print(f"[OK] Top barcodes = {len(barcodes)}")

# df_top end_day는 TEXT(YYYYMMDD) → fct_detail DATE(YYYY-MM-DD)
TARGET_END_DAY_TEXT = str(df_top["end_day"].iloc[0]).strip()
TARGET_END_DAY_DATE = pd.to_datetime(TARGET_END_DAY_TEXT, format="%Y%m%d", errors="raise").strftime("%Y-%m-%d")
TARGET_REMARK = str(df_top["remark"].iloc[0]).strip() if "remark" in df_top.columns else "PD"

print(f"[OK] TARGET_END_DAY_TEXT={TARGET_END_DAY_TEXT} / TARGET_END_DAY_DATE={TARGET_END_DAY_DATE} / REMARK={TARGET_REMARK}")

SQL_FCT_DETAIL = text("""
SELECT
    barcode_information,
    remark,
    end_day,
    end_time,
    contents,
    test_ct,
    test_time
FROM c1_fct_detail.fct_detail
WHERE end_day = CAST(:end_day AS date)
  AND remark = :remark
  AND barcode_information = ANY(CAST(:barcodes AS text[]))
""")

df_detail = pd.read_sql(
    SQL_FCT_DETAIL,
    engine,
    params={"end_day": TARGET_END_DAY_DATE, "remark": TARGET_REMARK, "barcodes": barcodes}
)

# ✅ df_top에서 station/run_time/boundary_run_time 붙이기(barcode 기준 1행)
df_meta = df_top[["barcode_information", "station", "run_time", "boundary_run_time"]].copy()
df_meta["barcode_information"] = df_meta["barcode_information"].astype(str)
df_meta["station"] = df_meta["station"].astype(str)
df_meta["run_time"] = pd.to_numeric(df_meta["run_time"], errors="coerce")
df_meta["boundary_run_time"] = pd.to_numeric(df_meta["boundary_run_time"], errors="coerce")
df_meta = df_meta.drop_duplicates("barcode_information")

df_detail["barcode_information"] = df_detail["barcode_information"].astype(str)
df_detail = df_detail.merge(df_meta, on="barcode_information", how="left")

# 컬럼 순서(요구: barcode 다음 station, 그리고 end_time 다음 run_time)
df_detail = df_detail[[
    "barcode_information",
    "station",
    "remark",
    "end_day",
    "end_time",
    "run_time",
    "boundary_run_time",
    "contents",
    "test_ct",
    "test_time"
]].copy()

print(f"[OK] df_detail rows={len(df_detail)}")

df_detail = _reorder_run_time_after_end_time(df_detail, "run_time", "end_time")
display_scroll_df(
    df_detail, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "test_ct":"{:.3f}"},
    title="df_detail (top 300, scroll)"
)

[OK] Top barcodes = 28
[OK] TARGET_END_DAY_TEXT=20251219 / TARGET_END_DAY_DATE=2025-12-19 / REMARK=PD
[OK] df_detail rows=9371


,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,contents,test_ct,test_time
0,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,3~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~END,,00:19:54.62
1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00674~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~00~OK~END,0.020,00:19:54.64
2,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: DMM SET CURRENT RANGE,1.860,00:19:56.50
3,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,MODE SET :: CURR_RANG,0.040,00:19:56.54
4,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,테스트 결과 : OK,0.210,00:19:56.75
5,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: DO SET,0.010,00:19:56.76
6,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,DO_SET Value :: 0900800080005000,0.010,00:19:56.77
7,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,테스트 결과 : OK,0.020,00:19:56.79
8,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: LOAD_C SET CC MODE,0.010,00:19:56.80
9,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,Return Value :: OK,0.010,00:19:56.81


In [7]:
# ============================================
# Cell X2) group 생성 + 정렬 + group 제외 규칙 적용
#  - group_key: barcode_information + end_day + end_time (station 제외)
# ============================================

import pandas as pd

df2 = df_detail.copy()

df2["barcode_information"] = df2["barcode_information"].astype(str)
df2["station"] = df2["station"].astype(str)
df2["remark"] = df2["remark"].astype(str)
df2["end_day"] = pd.to_datetime(df2["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
df2["end_time"] = df2["end_time"].astype(str)

def parse_test_time_to_ts(end_day_str, t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return pd.NaT
    s = str(t).strip()
    if s == "" or s.lower() == "none":
        return pd.NaT
    return pd.to_datetime(f"{end_day_str} {s}", errors="coerce")

df2["_test_ts"] = [parse_test_time_to_ts(d, t) for d, t in zip(df2["end_day"], df2["test_time"])]

df2["group_key"] = df2["barcode_information"] + "|" + df2["end_day"] + "|" + df2["end_time"]
df2["group"] = pd.factorize(df2["group_key"], sort=False)[0] + 1

# 제외 (6)
skip_mask = df2["contents"].astype(str).str.contains("START :: MES 이전공정 체크 SKIP", na=False)
skip_groups = set(df2.loc[skip_mask, "group"].unique().tolist())

# (7)(8)
df2_sorted_tmp = df2.sort_values(["group", "_test_ts"], ascending=[True, True]).copy()

def first_3_row(sub):
    m = sub["contents"].astype(str).str.startswith("3~", na=False)
    if not m.any():
        return None
    return sub.loc[m].iloc[0]

valid_groups = []
for g, sub in df2_sorted_tmp.groupby("group", sort=False):
    if g in skip_groups:
        continue
    r = first_3_row(sub)
    if r is None:
        continue
    if pd.isna(r["test_ct"]):
        valid_groups.append(g)

df2 = df2[df2["group"].isin(valid_groups)].copy()

df2["_is_first_3_null"] = (
    df2["contents"].astype(str).str.startswith("3~", na=False) &
    df2["test_ct"].isna()
).astype(int)

df2 = df2.sort_values(["group", "_is_first_3_null", "_test_ts"], ascending=[True, False, True]).reset_index(drop=True)
df2.drop(columns=["group_key"], inplace=True, errors="ignore")

print(f"[OK] df2 rows={len(df2)} / groups={df2['group'].nunique()}")

df2 = _reorder_run_time_after_end_time(df2, "run_time", "end_time")
display_scroll_df(
    df2, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "test_ct":"{:.3f}"},
    title="df2 (top 300, scroll)"
)

[OK] df2 rows=9371 / groups=40


,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,contents,test_ct,test_time,_test_ts,group,_is_first_3_null
0,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,3~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~END,,00:19:54.62,2025-12-19 00:19:54.620000,1,1
1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00674~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~00~OK~END,0.020,00:19:54.64,2025-12-19 00:19:54.640000,1,0
2,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: DMM SET CURRENT RANGE,1.860,00:19:56.50,2025-12-19 00:19:56.500000,1,0
3,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,MODE SET :: CURR_RANG,0.040,00:19:56.54,2025-12-19 00:19:56.540000,1,0
4,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,테스트 결과 : OK,0.210,00:19:56.75,2025-12-19 00:19:56.750000,1,0
5,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: DO SET,0.010,00:19:56.76,2025-12-19 00:19:56.760000,1,0
6,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,DO_SET Value :: 0900800080005000,0.010,00:19:56.77,2025-12-19 00:19:56.770000,1,0
7,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,테스트 결과 : OK,0.020,00:19:56.79,2025-12-19 00:19:56.790000,1,0
8,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: LOAD_C SET CC MODE,0.010,00:19:56.80,2025-12-19 00:19:56.800000,1,0
9,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,Return Value :: OK,0.010,00:19:56.81,2025-12-19 00:19:56.810000,1,0


In [8]:
# ============================================
# Cell X3) from_to_test_ct 생성 (OK/NG만)
# ============================================

import pandas as pd

df3 = df2.copy()
df3["_base_ts"] = df3.groupby("group")["_test_ts"].transform("first")

is_okng = df3["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df3["from_to_test_ct"] = pd.NA
mask = is_okng & df3["_test_ts"].notna() & df3["_base_ts"].notna()
df3.loc[mask, "from_to_test_ct"] = (df3.loc[mask, "_test_ts"] - df3.loc[mask, "_base_ts"]).dt.total_seconds()

print(f"[OK] from_to_test_ct filled rows={df3['from_to_test_ct'].notna().sum()}")

df3 = _reorder_run_time_after_end_time(df3, "run_time", "end_time")
display_scroll_df(
    df3, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "from_to_test_ct":"{:.3f}"},
    title="df3 (top 300, scroll)"
)

[OK] from_to_test_ct filled rows=2990


,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,contents,test_ct,test_time,_test_ts,group,_is_first_3_null,_base_ts,from_to_test_ct
0,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,3~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~END,,00:19:54.62,2025-12-19 00:19:54.620000,1,1,2025-12-19 00:19:54.620000,
1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00674~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~00~OK~END,0.020000,00:19:54.64,2025-12-19 00:19:54.640000,1,0,2025-12-19 00:19:54.620000,
2,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: DMM SET CURRENT RANGE,1.860000,00:19:56.50,2025-12-19 00:19:56.500000,1,0,2025-12-19 00:19:54.620000,
3,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,MODE SET :: CURR_RANG,0.040000,00:19:56.54,2025-12-19 00:19:56.540000,1,0,2025-12-19 00:19:54.620000,
4,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,테스트 결과 : OK,0.210000,00:19:56.75,2025-12-19 00:19:56.750000,1,0,2025-12-19 00:19:54.620000,2.130
5,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: DO SET,0.010000,00:19:56.76,2025-12-19 00:19:56.760000,1,0,2025-12-19 00:19:54.620000,
6,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,DO_SET Value :: 0900800080005000,0.010000,00:19:56.77,2025-12-19 00:19:56.770000,1,0,2025-12-19 00:19:54.620000,
7,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,테스트 결과 : OK,0.020000,00:19:56.79,2025-12-19 00:19:56.790000,1,0,2025-12-19 00:19:54.620000,2.170
8,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,START :: LOAD_C SET CC MODE,0.010000,00:19:56.80,2025-12-19 00:19:56.800000,1,0,2025-12-19 00:19:54.620000,
9,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,Return Value :: OK,0.010000,00:19:56.81,2025-12-19 00:19:56.810000,1,0,2025-12-19 00:19:54.620000,


In [9]:
# ============================================
# Cell X4) okng_seq + test_contents 매핑 + df_final 출력
# ============================================

import pandas as pd

df4 = df3.copy()

MAP_1_78 = {  # (생략 없이 그대로 유지)
  1:"1.00_dmm_c_rng_set", 2:"1.00_d_sig_val_090_set", 3:"1.00_load_c_set_cc", 4:"1.00_load_c_cc_rng_set",
  5:"1.00_d_sig_val_000_set", 6:"1.00_dmm_dc_v_set", 7:"1.00_dmm_ac_0.6_set", 8:"1.00_ps_14.7_set",
  9:"1.00_dmm_dc_c_set", 10:"1.01_ps_14.7_on", 11:"1.00_dmm_ac_0.6_set", 12:"1.01_input_14.7v",
  13:"1.02_usb_c_pm1", 14:"1.03_usb_c_pm2", 15:"1.04_usb_c_pm3", 16:"1.05_usb_c_pm4",
  17:"1.06_usb_a_pm1", 18:"1.07_usb_a_pm2", 19:"1.08_usb_a_pm3", 20:"1.09_usb_a_pm4",
  21:"1.10_fw_ver_check", 22:"1.11_chip_id_check", 23:"1.12_usb_c_carplay", 24:"1.13_usb_a_carplay",
  25:"1.14_pd_profile_count", 26:"1.15_dmm_c_rng_set", 27:"1.15_load_a_cc_set", 28:"1.15_load_a_rng_set",
  29:"1.15_load_c_cc_set", 30:"1.15_load_c_rng_set", 31:"1.15_dmm_regi_set", 32:"1.15_dmm_regi_ac_0.6_set",
  33:"1.15_d_sig_val_000_set", 34:"1.15_pin12_short_check", 35:"1.16_pin23_short_check", 36:"1.17_pin34_short_check",
  37:"1.18_dmm_dc_v_set", 38:"1.18_dmm_ac_0.6_set", 39:"1.18_dmm_dc_c_set", 40:"1.18_load_a_sensing_on",
  41:"1.18_load_c_sensing_on", 42:"1.18_ps_18v_set", 43:"1.18_ps_18v_on", 44:"1.18_dmm_ac_0.6_set",
  45:"1.18_input_18v", 46:"1.19_idle_c_check", 47:"1.20_no_load_usb_c", 48:"1.21_no_load_usb_a",
  49:"1.22_dmm_3c_rng_set", 50:"1.22_load_a_5c_set", 51:"1.22_load_a_on", 52:"1.22_overcurr_usb_a_c",
  53:"1.23_overcurr_usb_a_v", 54:"1.24_usb_c_v", 55:"1.25_load_a_off", 56:"1.25_load_c_5c_set",
  57:"1.24_load_c_on", 58:"1.25_overcurr_usb_c_c", 59:"1.26_overcurr_usb_c_v", 60:"1.27_usb_a_v",
  61:"1.28_load_c_off", 62:"1.28_load_a_2.4c_set", 63:"1.28_load_c_3c_set", 64:"1.28_load_a_2.4c_on",
  65:"1.28_load_c_3c_on", 66:"1.28_usb_c_bside_3c_check", 67:"1.29_usb_c_bside_v_check", 68:"1.30_usb_a_2.4c_check",
  69:"1.31_usb_a_v_check", 70:"1.32_load_c_1.3c_set", 71:"1.32_pdo4_set", 72:"1.33_usb_c_1.35c_check",
  73:"1.34_usb_c_v_check", 74:"1.35_usb_c_aside_cc_check", 75:"1.36_load_c_off", 76:"1.36_dmm_ac_0.6_set",
  77:"1.36_dmm_c_rng_set", 78:"1.36_dark_curr_check",
}

is_okng = df4["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df4["okng_seq"] = pd.NA
df4.loc[is_okng, "okng_seq"] = (
    df4.loc[is_okng]
       .groupby("group")
       .cumcount()
       .add(1)
       .astype(int)
)

df4["test_contents"] = pd.NA
df4.loc[is_okng, "test_contents"] = df4.loc[is_okng, "okng_seq"].map(MAP_1_78)

df_final = df4[[
    "group",
    "barcode_information",
    "station",
    "remark",
    "end_day",
    "end_time",
    "run_time",
    "boundary_run_time",
    "test_time",
    "contents",
    "test_contents",
    "test_ct",
    "from_to_test_ct",
    "okng_seq"
]].copy()

print(f"[OK] df_final rows={len(df_final)} / groups={df_final['group'].nunique()}")

df_final = _reorder_run_time_after_end_time(df_final, "run_time", "end_time")
display_scroll_df(
    df_final, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "test_ct":"{:.3f}", "from_to_test_ct":"{:.3f}"},
    title="df_final (top 300, scroll)"
)

[OK] df_final rows=9371 / groups=40


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,test_time,contents,test_contents,test_ct,from_to_test_ct,okng_seq
0,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:54.62,3~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~END,,,,
1,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:54.64,00674~192.168.108.154~FCT~PPIDBA1WJ25350502034USJ8T-14F014-AF~00~OK~END,,0.020,,
2,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.50,START :: DMM SET CURRENT RANGE,,1.860,,
3,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.54,MODE SET :: CURR_RANG,,0.040,,
4,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.75,테스트 결과 : OK,1.00_dmm_c_rng_set,0.210,2.130,1
5,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.76,START :: DO SET,,0.010,,
6,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.77,DO_SET Value :: 0900800080005000,,0.010,,
7,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.79,테스트 결과 : OK,1.00_d_sig_val_090_set,0.020,2.170,2
8,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.80,START :: LOAD_C SET CC MODE,,0.010,,
9,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.81,Return Value :: OK,,0.010,,


In [10]:
# ============================================
# Cell X5) df15 생성 (contents/test_ct 제외, test_contents NA 제외)
# ============================================

import pandas as pd

df15 = df_final.dropna(subset=["test_contents"]).copy()

df15 = df15[[
    "group",
    "barcode_information",
    "station",
    "remark",
    "end_day",
    "end_time",
    "run_time",
    "boundary_run_time",
    "test_time",
    "test_contents",
    "from_to_test_ct",
    "okng_seq"
]].reset_index(drop=True)

print(f"[OK] df15 rows={len(df15)} / groups={df15['group'].nunique()}")

df15 = _reorder_run_time_after_end_time(df15, "run_time", "end_time")
display_scroll_df(
    df15, n=300, height_px=560,
    float_format={"run_time":"{:.2f}", "boundary_run_time":"{:.2f}", "from_to_test_ct":"{:.3f}"},
    title="df15 (top 300, scroll)"
)

[OK] df15 rows=2990 / groups=40


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,test_time,test_contents,from_to_test_ct,okng_seq
0,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.75,1.00_dmm_c_rng_set,2.130,1
1,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.79,1.00_d_sig_val_090_set,2.170,2
2,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.82,1.00_load_c_set_cc,2.200,3
3,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.86,1.00_load_c_cc_rng_set,2.240,4
4,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:56.89,1.00_d_sig_val_000_set,2.270,5
5,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:57.09,1.00_dmm_dc_v_set,2.470,6
6,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:57.34,1.00_dmm_ac_0.6_set,2.720,7
7,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:57.49,1.00_ps_14.7_set,2.870,8
8,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:57.74,1.00_dmm_dc_c_set,3.120,9
9,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,00:19:58.15,1.01_ps_14.7_on,3.530,10


In [11]:
# ============================================
# Cell X6) df15에 summary upper_outlier + problem1~3 JOIN
# 1) boundary_run_time_summary -> boundary_test_ct 로 컬럼명 변경
# 2) 출력 컬럼 순서 고정:
#    group, barcode_information, station, remark, end_day, end_time, run_time,
#    boundary_run_time, okng_seq, test_contents, test_time, from_to_test_ct,
#    boundary_test_ct, problem1, problem2, problem3
#  - 출력: 300행, 상하/좌우 스크롤
# ============================================

import pandas as pd
from sqlalchemy import text

if "df15" not in globals() or len(df15) == 0:
    raise ValueError("[ERROR] df15가 없거나 비어있습니다. Cell X5를 먼저 실행하세요.")

TARGET_END_DAY = str(df15["end_day"].dropna().iloc[0]).strip()

# problem1~3 존재 확인
SQL_PROBLEM_COLS = text("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'e4_predictive_maintenance'
  AND table_name   = 'pd_cal_test_ct_summary'
  AND column_name IN ('problem1','problem2','problem3','problem4')
ORDER BY column_name
""")
present_problem_cols = pd.read_sql(SQL_PROBLEM_COLS, engine)["column_name"].tolist()
present_problem_cols = [c.strip() for c in present_problem_cols if isinstance(c, str)]
want_problem_cols = ["problem1", "problem2", "problem3", "problem4"]
use_problem_cols = [c for c in want_problem_cols if c in present_problem_cols]

# summary day 매칭/폴백(TEXT 비교)
SQL_HAS_DAY = text("""
SELECT COUNT(*) AS n
FROM e4_predictive_maintenance.pd_cal_test_ct_summary
WHERE end_day = :end_day
""")
n = int(pd.read_sql(SQL_HAS_DAY, engine, params={"end_day": TARGET_END_DAY}).iloc[0]["n"])

if n > 0:
    SUMMARY_DAY = TARGET_END_DAY
else:
    SQL_LATEST_DAY = text("""
    SELECT MAX(end_day) AS max_day
    FROM e4_predictive_maintenance.pd_cal_test_ct_summary
    """)
    SUMMARY_DAY = pd.read_sql(SQL_LATEST_DAY, engine).iloc[0]["max_day"]
    if pd.isna(SUMMARY_DAY):
        raise ValueError("[ERROR] pd_cal_test_ct_summary 테이블이 비어있습니다.")
    SUMMARY_DAY = str(SUMMARY_DAY).strip()

# boundary + problem1~3 로드
select_cols = ["test_contents", "upper_outlier"] + use_problem_cols
SQL_BOUNDARY = text(f"""
SELECT {", ".join(select_cols)}
FROM e4_predictive_maintenance.pd_cal_test_ct_summary
WHERE end_day = :end_day
""")

df_boundary = pd.read_sql(SQL_BOUNDARY, engine, params={"end_day": SUMMARY_DAY})
df_boundary["test_contents"] = df_boundary["test_contents"].astype(str).str.strip()
df_boundary["upper_outlier"] = pd.to_numeric(df_boundary["upper_outlier"], errors="coerce")

for c in use_problem_cols:
    df_boundary[c] = df_boundary[c].astype(str).str.strip()

# JOIN
df15_out = df15.copy()
df15_out["test_contents"] = df15_out["test_contents"].astype(str).str.strip()

join_cols = ["test_contents", "upper_outlier"] + use_problem_cols
df15_out = df15_out.merge(df_boundary[join_cols], on="test_contents", how="left")

# 1) 컬럼명 변경: upper_outlier -> boundary_test_ct
df15_out = df15_out.rename(columns={"upper_outlier": "boundary_test_ct"})

# problem1~3 컬럼 보장
for c in want_problem_cols:
    if c not in df15_out.columns:
        df15_out[c] = pd.NA

# 2) 출력 컬럼 순서 고정 (요구사항)
OUT_COLS_X6 = [
    "group","barcode_information","station","remark","end_day","end_time","run_time",
    "boundary_run_time","okng_seq","test_contents","test_time","from_to_test_ct",
    "boundary_test_ct","problem1","problem2","problem3","problem4"
]
# 누락 컬럼 체크(있으면 유지, 없으면 에러를 명확히)
missing = [c for c in OUT_COLS_X6 if c not in df15_out.columns]
if missing:
    raise KeyError(f"[ERROR] df15_out에 필요한 컬럼이 없습니다: {missing}")

df15_out = df15_out[OUT_COLS_X6].copy()

print(f"[OK] df15_out rows={len(df15_out)} / boundary_test_ct filled={df15_out['boundary_test_ct'].notna().sum()}")

# 스크롤 출력(300행)
display_scroll_df(
    df15_out,
    n=300,
    height_px=560,
    float_format={
        "run_time":"{:.2f}",
        "boundary_run_time":"{:.2f}",
        "boundary_test_ct":"{:.2f}",
        "from_to_test_ct":"{:.3f}",
    },
    title="Cell X6: df15_out (top 300, scroll)"
)

[OK] df15_out rows=3150 / boundary_test_ct filled=3150


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,problem1,problem2,problem3,problem4
0,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,1,1.00_dmm_c_rng_set,00:19:56.75,2.130,2.99,dmm,relay_board,None,None
1,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,2,1.00_d_sig_val_090_set,00:19:56.79,2.170,3.07,relay_board,None,None,None
2,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,3,1.00_load_c_set_cc,00:19:56.82,2.200,3.10,load_c,relay_board,None,None
3,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,4,1.00_load_c_cc_rng_set,00:19:56.86,2.240,3.14,load_c,relay_board,None,None
4,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,5,1.00_d_sig_val_000_set,00:19:56.89,2.270,3.20,relay_board,None,None,None
5,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,6,1.00_dmm_dc_v_set,00:19:57.09,2.470,3.37,dmm,relay_board,None,None
6,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,7,1.00_dmm_ac_0.6_set,00:19:57.34,2.720,5.53,dmm,relay_board,None,None
7,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,7,1.00_dmm_ac_0.6_set,00:19:57.34,2.720,5.53,dmm,relay_board,None,None
8,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,8,1.00_ps_14.7_set,00:19:57.49,2.870,3.77,power_supply,dmm,relay_board,None
9,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,9,1.00_dmm_dc_c_set,00:19:57.74,3.120,4.01,dmm,relay_board,None,None


In [12]:
# ============================================
# Cell X7) diff_ct 생성 (수식 수정)
# - diff_ct = from_to_test_ct - boundary_test_ct
# - 컬럼 순서 고정 출력
# ============================================

import pandas as pd

if "df15_out" not in globals() or len(df15_out) == 0:
    raise ValueError("[ERROR] df15_out가 없거나 비어있습니다. Cell X6를 먼저 실행하세요.")

work = df15_out.copy()

work["boundary_test_ct"] = pd.to_numeric(work["boundary_test_ct"], errors="coerce")
work["from_to_test_ct"]  = pd.to_numeric(work["from_to_test_ct"], errors="coerce")

# ✅ 수식 수정
work["diff_ct"] = work["from_to_test_ct"] - work["boundary_test_ct"]

OUT_COLS_X7 = [
    "group","barcode_information","station","remark","end_day","end_time","run_time",
    "boundary_run_time","okng_seq","test_contents","test_time","from_to_test_ct",
    "boundary_test_ct","diff_ct","problem1","problem2","problem3","problem4"
]
missing = [c for c in OUT_COLS_X7 if c not in work.columns]
if missing:
    raise KeyError(f"[ERROR] X7 출력에 필요한 컬럼이 없습니다: {missing}")

df15_out2 = work[OUT_COLS_X7].copy()

print(f"[OK] df15_out2 rows={len(df15_out2)} / diff_ct non-null={df15_out2['diff_ct'].notna().sum()}")

display_scroll_df(
    df15_out2,
    n=300,
    height_px=560,
    float_format={
        "run_time":"{:.2f}",
        "boundary_run_time":"{:.2f}",
        "boundary_test_ct":"{:.2f}",
        "from_to_test_ct":"{:.3f}",
        "diff_ct":"{:.3f}",
    },
    title="Cell X7: df15_out2 (diff_ct = from_to_test_ct - boundary_test_ct)"
)

[OK] df15_out2 rows=3150 / diff_ct non-null=3150


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,diff_ct,problem1,problem2,problem3,problem4
0,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,1,1.00_dmm_c_rng_set,00:19:56.75,2.130,2.99,-0.860,dmm,relay_board,None,None
1,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,2,1.00_d_sig_val_090_set,00:19:56.79,2.170,3.07,-0.900,relay_board,None,None,None
2,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,3,1.00_load_c_set_cc,00:19:56.82,2.200,3.10,-0.900,load_c,relay_board,None,None
3,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,4,1.00_load_c_cc_rng_set,00:19:56.86,2.240,3.14,-0.900,load_c,relay_board,None,None
4,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,5,1.00_d_sig_val_000_set,00:19:56.89,2.270,3.20,-0.930,relay_board,None,None,None
5,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,6,1.00_dmm_dc_v_set,00:19:57.09,2.470,3.37,-0.900,dmm,relay_board,None,None
6,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,7,1.00_dmm_ac_0.6_set,00:19:57.34,2.720,5.53,-2.810,dmm,relay_board,None,None
7,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,7,1.00_dmm_ac_0.6_set,00:19:57.34,2.720,5.53,-2.810,dmm,relay_board,None,None
8,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,8,1.00_ps_14.7_set,00:19:57.49,2.870,3.77,-0.900,power_supply,dmm,relay_board,None
9,1,BA1WJ25350502034USJ8T-14F014-AF,FCT4,PD,2025-12-19,00:20:36,41.80,33.19,9,1.00_dmm_dc_c_set,00:19:57.74,3.120,4.01,-0.890,dmm,relay_board,None,None


In [16]:
# ============================================
# Cell X8) diff_ct Worst TOP 5% (정상 구조)
# - 원본 전체에서 diff_ct 기준 TOP5% 개수만큼 먼저 자른다
# - 그 TOP5% 안에서 barcode 중복 제거 (okng_seq 가장 빠른 것만 유지)
# - run_time 내림차순 정렬
# ============================================

import math
import pandas as pd

if "df15_out2" not in globals() or len(df15_out2) == 0:
    raise ValueError("[ERROR] df15_out2가 없거나 비어있습니다. Cell X7을 먼저 실행하세요.")

df = df15_out2.copy()

df["diff_ct"]  = pd.to_numeric(df["diff_ct"], errors="coerce")
df["run_time"] = pd.to_numeric(df["run_time"], errors="coerce")
df["okng_seq"] = pd.to_numeric(df["okng_seq"], errors="coerce")

df = df.dropna(subset=["barcode_information","diff_ct","okng_seq"]).copy()

# 1️⃣ 원본 전체에서 TOP5% 개수만큼 먼저 컷
n_all = len(df)
top_n = math.ceil(n_all * 0.05)

df_top_raw = df.sort_values("diff_ct", ascending=False).head(top_n).copy()

# 2️⃣ TOP5% 안에서 barcode 대표화 (okng_seq 최소값 1건)
df_top_raw["min_okng"] = df_top_raw.groupby("barcode_information")["okng_seq"].transform("min")

df_spike = (
    df_top_raw[df_top_raw["okng_seq"] == df_top_raw["min_okng"]]
        .sort_values(["barcode_information","diff_ct","run_time"], ascending=[True,False,False])
        .drop_duplicates("barcode_information")
        .sort_values("run_time", ascending=False)
        .reset_index(drop=True)
)

OUT_COLS_X8 = [
    "group","barcode_information","station","remark","end_day","end_time","run_time",
    "boundary_run_time","okng_seq","test_contents","test_time","from_to_test_ct",
    "boundary_test_ct","diff_ct","problem1","problem2","problem3","problem4"
]
missing = [c for c in OUT_COLS_X8 if c not in df_spike.columns]
if missing:
    raise KeyError(f"[ERROR] X8 출력에 필요한 컬럼이 없습니다: {missing}")

df_spike = df_spike[OUT_COLS_X8].copy()

print(f"[OK] 원본={n_all} → TOP5% raw={len(df_top_raw)} → 최종 worst rows={len(df_spike)}")

display_scroll_df(
    df_spike,
    n=300,
    height_px=560,
    float_format={
        "run_time":"{:.2f}",
        "boundary_run_time":"{:.2f}",
        "boundary_test_ct":"{:.2f}",
        "from_to_test_ct":"{:.3f}",
        "diff_ct":"{:.3f}",
    },
    title="Cell X8: diff_ct Worst TOP 5% (dedup by barcode after cut)"
)

[OK] 원본=3150 → TOP5% raw=158 → 최종 worst rows=8


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,diff_ct,problem1,problem2,problem3,problem4
0,4,BA1WJ25350501252USJ8T-14F014-AF,FCT1,PD,2025-12-19,00:34:32,51.80,33.19,57,1.24_load_c_on,00:34:25.53,46.590,23.13,23.460,load_c,relay_board,None,None
1,36,BA1WJ25352501392USJ8T-14F014-AF,FCT1,PD,2025-12-19,08:48:11,48.70,33.19,30,1.15_load_c_rng_set,08:47:55.64,33.420,15.79,17.630,load_c,relay_board,None,None
2,15,BA1WJ25352508002USJ8T-14F014-AF,FCT4,PD,2025-12-19,05:33:27,47.70,33.19,66,1.28_usb_c_bside_3c_check,05:33:27.38,47.700,28.16,19.540,usb_c,dmm,relay_board,load_c
3,9,BA1WJ25351504376USJ8T-14F014-AF,FCT1,PD,2025-12-19,02:10:00,47.30,33.19,30,1.15_load_c_rng_set,02:09:45.53,33.300,15.79,17.510,load_c,relay_board,None,None
4,5,BA1WJ25351505288USJ8T-14F014-AF,FCT1,PD,2025-12-19,01:09:41,47.20,33.19,63,1.28_load_c_3c_set,01:09:35.41,42.090,26.42,15.670,load_c,dmm,relay_board,None
5,21,BA1WJ25352502383USJ8T-14F014-AF,FCT1,PD,2025-12-19,06:40:58,46.60,33.19,53,1.23_overcurr_usb_a_v,06:40:50.54,39.960,22.77,17.190,usb_a,dmm,relay_board,load_a
6,8,BA1WJ25352501264USJ8T-14F014-AF,FCT1,PD,2025-12-19,03:19:09,46.00,33.19,53,1.23_overcurr_usb_a_v,03:19:02.38,40.410,22.77,17.640,usb_a,dmm,relay_board,load_a
7,31,BA1WJ25352507569USJ8T-14F014-AF,FCT1,PD,2025-12-19,07:32:37,45.90,33.19,64,1.28_load_a_2.4c_on,07:32:32.44,41.560,26.58,14.980,load_a,dmm,relay_board,None


In [17]:
# ============================================
# Cell X9) df_spike(X8) 키(barcode_information,end_day,end_time)로
#          c1_fct_detail.fct_detail 매칭 → file_path 컬럼 추가
#  - 결과: df_spike_fp
# ============================================

import pandas as pd
from sqlalchemy import text

if "df_spike" not in globals() or len(df_spike) == 0:
    raise ValueError("[ERROR] df_spike가 없거나 비어있습니다. Cell X8을 먼저 실행하세요.")

# 매칭 키 정리
key_df = df_spike[["barcode_information", "end_day", "end_time"]].copy()
key_df["barcode_information"] = key_df["barcode_information"].astype(str).str.strip()
key_df["end_day"] = pd.to_datetime(key_df["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
key_df["end_time"] = key_df["end_time"].astype(str).str.strip()

barcodes = key_df["barcode_information"].dropna().drop_duplicates().tolist()
days     = key_df["end_day"].dropna().drop_duplicates().tolist()

if len(barcodes) == 0 or len(days) == 0:
    raise ValueError("[ERROR] df_spike에서 barcode/end_day를 추출하지 못했습니다.")

print(f"[OK] X9 match keys: barcodes={len(barcodes)}, days={len(days)}")

# fct_detail에서 필요한 키 + file_path만 로드 (end_time을 text로 통일)
SQL_FILEPATH = text("""
SELECT
    barcode_information::text AS barcode_information,
    to_char(end_day, 'YYYY-MM-DD') AS end_day,
    end_time::text AS end_time,
    file_path::text AS file_path
FROM c1_fct_detail.fct_detail
WHERE barcode_information = ANY(CAST(:barcodes AS text[]))
  AND end_day = ANY(CAST(:days AS date[]))
  AND file_path IS NOT NULL
""")

df_fp = pd.read_sql(SQL_FILEPATH, engine, params={"barcodes": barcodes, "days": days})

# 키 정규화
df_fp["barcode_information"] = df_fp["barcode_information"].astype(str).str.strip()
df_fp["end_day"] = pd.to_datetime(df_fp["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
df_fp["end_time"] = df_fp["end_time"].astype(str).str.strip()
df_fp["file_path"] = df_fp["file_path"].astype(str)

# (barcode,end_day,end_time) 중복 발생 시 file_path 1개로 압축
df_fp = (
    df_fp.dropna(subset=["barcode_information", "end_day", "end_time"])
         .sort_values(["barcode_information", "end_day", "end_time"])
         .drop_duplicates(["barcode_information", "end_day", "end_time"], keep="first")
)

# df_spike에 병합
df_spike_fp = df_spike.copy()
df_spike_fp["barcode_information"] = df_spike_fp["barcode_information"].astype(str).str.strip()
df_spike_fp["end_day"] = pd.to_datetime(df_spike_fp["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
df_spike_fp["end_time"] = df_spike_fp["end_time"].astype(str).str.strip()

df_spike_fp = df_spike_fp.merge(
    df_fp[["barcode_information", "end_day", "end_time", "file_path"]],
    on=["barcode_information", "end_day", "end_time"],
    how="left"
)

# file_path를 맨 뒤에 추가(원하면 위치 변경 가능)
print(f"[OK] df_spike_fp rows={len(df_spike_fp)} / file_path filled={df_spike_fp['file_path'].notna().sum()}")

display_scroll_df(
    df_spike_fp,
    n=300,
    height_px=560,
    float_format={
        "run_time":"{:.2f}",
        "boundary_run_time":"{:.2f}",
        "boundary_test_ct":"{:.2f}",
        "from_to_test_ct":"{:.3f}",
        "diff_ct":"{:.3f}",
    },
    title="Cell X9: df_spike_fp (+file_path)"
)

[OK] X9 match keys: barcodes=8, days=1
[OK] df_spike_fp rows=8 / file_path filled=8


,group,barcode_information,station,remark,end_day,end_time,run_time,boundary_run_time,okng_seq,test_contents,test_time,from_to_test_ct,boundary_test_ct,diff_ct,problem1,problem2,problem3,problem4,file_path
0,4,BA1WJ25350501252USJ8T-14F014-AF,FCT1,PD,2025-12-19,00:34:32,51.80,33.19,57,1.24_load_c_on,00:34:25.53,46.590,23.13,23.460,load_c,relay_board,None,None,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25350501252USJ8T-14F014-AF_20251219_003338.txt
1,36,BA1WJ25352501392USJ8T-14F014-AF,FCT1,PD,2025-12-19,08:48:11,48.70,33.19,30,1.15_load_c_rng_set,08:47:55.64,33.420,15.79,17.630,load_c,relay_board,None,None,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25352501392USJ8T-14F014-AF_20251219_084721.txt
2,15,BA1WJ25352508002USJ8T-14F014-AF,FCT4,PD,2025-12-19,05:33:27,47.70,33.19,66,1.28_usb_c_bside_3c_check,05:33:27.38,47.700,28.16,19.540,usb_c,dmm,relay_board,load_c,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25352508002USJ8T-14F014-AF_20251219_053239.txt
3,9,BA1WJ25351504376USJ8T-14F014-AF,FCT1,PD,2025-12-19,02:10:00,47.30,33.19,30,1.15_load_c_rng_set,02:09:45.53,33.300,15.79,17.510,load_c,relay_board,None,None,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25351504376USJ8T-14F014-AF_20251219_020911.txt
4,5,BA1WJ25351505288USJ8T-14F014-AF,FCT1,PD,2025-12-19,01:09:41,47.20,33.19,63,1.28_load_c_3c_set,01:09:35.41,42.090,26.42,15.670,load_c,dmm,relay_board,None,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25351505288USJ8T-14F014-AF_20251219_010852.txt
5,21,BA1WJ25352502383USJ8T-14F014-AF,FCT1,PD,2025-12-19,06:40:58,46.60,33.19,53,1.23_overcurr_usb_a_v,06:40:50.54,39.960,22.77,17.190,usb_a,dmm,relay_board,load_a,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25352502383USJ8T-14F014-AF_20251219_064010.txt
6,8,BA1WJ25352501264USJ8T-14F014-AF,FCT1,PD,2025-12-19,03:19:09,46.00,33.19,53,1.23_overcurr_usb_a_v,03:19:02.38,40.410,22.77,17.640,usb_a,dmm,relay_board,load_a,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25352501264USJ8T-14F014-AF_20251219_031821.txt
7,31,BA1WJ25352507569USJ8T-14F014-AF,FCT1,PD,2025-12-19,07:32:37,45.90,33.19,64,1.28_load_a_2.4c_on,07:32:32.44,41.560,26.58,14.980,load_a,dmm,relay_board,None,\\192.168.108.155\FCT LogFile\Machine Log\FCT\2025\12\19\FORD A+C PD FULL 35930928\BA1WJ25352507569USJ8T-14F014-AF_20251219_073150.txt


In [18]:
# ============================================
# Cell X10) df_spike_fp를 e4_predictive_maintenance.non_pd_worst 저장 (FIXED)
#  - 테이블 없으면 생성
#  - UPSERT(중복 시 업데이트)
#  - PK: (barcode_information, end_day, end_time, test_contents, okng_seq)
#  - ✅ group 바인드 에러 해결: group_no로 완전 치환
# ============================================

import pandas as pd
from sqlalchemy import text

if "df_spike_fp" not in globals() or len(df_spike_fp) == 0:
    raise ValueError("[ERROR] df_spike_fp가 없거나 비어있습니다. Cell X9를 먼저 실행하세요.")

TARGET_SCHEMA = "e4_predictive_maintenance"
TARGET_TABLE  = "pd_worst"
FULL_NAME     = f"{TARGET_SCHEMA}.{TARGET_TABLE}"

# -----------------------------
# 1) 저장 전 DF 정리
# -----------------------------
df_save = df_spike_fp.copy()

# group -> group_no 로 저장 컬럼 확정 (없으면 NULL)
if "group" in df_save.columns and "group_no" not in df_save.columns:
    df_save["group_no"] = pd.to_numeric(df_save["group"], errors="coerce").astype("Int64")
elif "group_no" not in df_save.columns:
    df_save["group_no"] = pd.NA

# 필수키 컬럼 존재 확인
pk_cols = ["barcode_information", "end_day", "end_time", "test_contents", "okng_seq"]
missing_pk = [c for c in pk_cols if c not in df_save.columns]
if missing_pk:
    raise KeyError(f"[ERROR] PK에 필요한 컬럼이 없습니다: {missing_pk}")

# 타입 정규화
df_save["barcode_information"] = df_save["barcode_information"].astype(str).str.strip()
df_save["end_day"] = pd.to_datetime(df_save["end_day"], errors="coerce").dt.date
df_save["end_time"] = df_save["end_time"].astype(str).str.strip()
df_save["test_contents"] = df_save["test_contents"].astype(str).str.strip()
df_save["okng_seq"] = pd.to_numeric(df_save["okng_seq"], errors="coerce").astype("Int64")

# 숫자형
for c in ["run_time","boundary_run_time","from_to_test_ct","boundary_test_ct","diff_ct"]:
    if c in df_save.columns:
        df_save[c] = pd.to_numeric(df_save[c], errors="coerce")

# problem 컬럼 보장
for c in ["problem1","problem2","problem3","problem4"]:
    if c not in df_save.columns:
        df_save[c] = pd.NA

# file_path 보장
if "file_path" not in df_save.columns:
    df_save["file_path"] = pd.NA

# -----------------------------
# 2) 테이블 생성(없으면)
# -----------------------------
DDL = text(f"""
CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA};

CREATE TABLE IF NOT EXISTS {FULL_NAME} (
    group_no            integer,
    barcode_information text NOT NULL,
    station             text,
    remark              text,
    end_day             date NOT NULL,
    end_time            text NOT NULL,
    run_time            double precision,
    boundary_run_time   double precision,
    okng_seq            integer NOT NULL,
    test_contents       text NOT NULL,
    test_time           text,
    from_to_test_ct     double precision,
    boundary_test_ct    double precision,
    diff_ct             double precision,
    problem1            text,
    problem2            text,
    problem3            text,
    problem4            text,
    file_path           text,
    updated_at          timestamp without time zone DEFAULT now(),
    PRIMARY KEY (barcode_information, end_day, end_time, test_contents, okng_seq)
);
""")

# -----------------------------
# 3) UPSERT SQL (✅ 바인드도 group_no로)
# -----------------------------
save_cols = [
    "group_no",
    "barcode_information","station","remark","end_day","end_time",
    "run_time","boundary_run_time",
    "okng_seq","test_contents","test_time",
    "from_to_test_ct","boundary_test_ct","diff_ct",
    "problem1","problem2","problem3","problem4",
    "file_path"
]
# 실제 존재 컬럼만
save_cols = [c for c in save_cols if c in df_save.columns]

insert_cols_sql = ", ".join(save_cols)
values_sql      = ", ".join([f":{c}" for c in save_cols])

conflict_cols = "barcode_information, end_day, end_time, test_contents, okng_seq"
set_cols = [c for c in save_cols if c not in pk_cols]  # 키 제외 업데이트
set_sql  = ", ".join([f"{c} = EXCLUDED.{c}" for c in set_cols] + ["updated_at = now()"])

UPSERT_SQL = text(f"""
INSERT INTO {FULL_NAME} ({insert_cols_sql})
VALUES ({values_sql})
ON CONFLICT ({conflict_cols})
DO UPDATE SET {set_sql};
""")

# -----------------------------
# 4) 실행
# -----------------------------
with engine.begin() as conn:
    conn.execute(DDL)

    # executemany 파라미터(dict) 생성
    rows = df_save[save_cols].to_dict(orient="records")

    # Int64/NaT/NaN 정리(선택: 안정성)
    for r in rows:
        # pandas NA -> None
        for k, v in list(r.items()):
            if pd.isna(v):
                r[k] = None

    conn.execute(UPSERT_SQL, rows)

print(f"[OK] Saved to {FULL_NAME} (rows={len(df_save)})")

[OK] Saved to e4_predictive_maintenance.pd_worst (rows=8)
